# 📊 05 — SETU-Rail AI/BI Dashboard

This notebook defines the SQL that powers each widget of our AI/BI dashboard. After running the cells to verify, you'll create the dashboard in the UI and embed it in our Databricks App.

In [0]:
%sql
-- Widget 1: KPI — Total trains in scope
SELECT COUNT(DISTINCT train_no) AS total_trains FROM setu_rail.silver.train_enriched;

In [0]:
%sql
-- Widget 2: KPI — Predicted avg delay
SELECT ROUND(AVG(predicted_delay_min), 1) AS avg_predicted_delay
FROM   setu_rail.gold.predictions_daily;

In [0]:
%sql
-- Widget 3: KPI — Cities with hazardous air
SELECT COUNT(*) AS hazardous_cities
FROM   setu_rail.gold.vw_top_polluted_routes
WHERE  avg_pm25 > 100;

In [0]:
%sql
-- Widget 4: Bar — Top polluted cities (fog proxy)
SELECT * FROM setu_rail.gold.vw_top_polluted_routes LIMIT 15;

In [0]:
%sql
-- Widget 5: Line — Avg delay by month (shows winter fog effect)
SELECT MONTH(run_date) AS m, ROUND(AVG(predicted_delay_min), 1) AS avg_delay
FROM   setu_rail.gold.predictions_daily
GROUP BY m
ORDER BY m;

In [0]:
%sql
-- Widget 6: Scatter — PM2.5 vs predicted delay (visualises fog-delay correlation)
SELECT pm25, predicted_delay_min
FROM   setu_rail.gold.predictions_daily
WHERE  pm25 IS NOT NULL
LIMIT 2000;

In [0]:
%sql
-- Widget 7: Bar — Hourly train schedule density
SELECT * FROM setu_rail.gold.vw_hourly_schedule;

In [0]:
%sql
-- Widget 8: Table — Top 10 most-delayed train-station pairs
SELECT train_no, station_code,
       ROUND(AVG(predicted_delay_min), 1) AS avg_delay
FROM   setu_rail.gold.predictions_daily
GROUP BY train_no, station_code
ORDER BY avg_delay DESC
LIMIT 10;

## UI steps

1. Left sidebar → **Dashboards** → **Create Dashboard** → name **"SETU-Rail Dashboard"**
2. On the **Data** tab, click **+ Create from SQL** and paste each query above — one dataset per widget
3. On the **Canvas** tab, add visualizations:

| Widget | Type | X / Category | Y / Value |
|---|---|---|---|
| Total trains | Counter | — | `total_trains` |
| Avg predicted delay | Counter | — | `avg_predicted_delay` |
| Hazardous cities | Counter | — | `hazardous_cities` |
| Top polluted cities | Bar | `city` | `avg_pm25` |
| Delay by month | Line | `m` | `avg_delay` |
| PM2.5 × Delay | Scatter | `pm25` | `predicted_delay_min` |
| Hourly schedule | Bar | `scheduled_hour` | `num_trains` |
| Top delayed pairs | Table | — | — |

4. Click **Publish**. Copy the **embed URL** (`/embed/dashboardsv3/...`) — you'll paste it into `app.yaml` in the next step.

✅ **Next:** `06_application/00_deploy_app.ipynb`